# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [5]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [7]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [9]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un Model Lingvistic Mare (LLM) este un tip de inteligență artificială care a fost antrenat pe cantități masive de text pentru a înțelege, genera și interacționa cu limbajul uman. Acești modele pot realiza diverse sarcini lingvistice, cum ar fi traducerea, rezumarea textelor, scrierea de povești sau răspunsuri la întrebări.
Un LLM (Large Language Model) este un tip de inteligență artificială antrenat pe cantități uriașe de text și date, permițându-i să înțeleagă, să genereze și să manipuleze limbajul uman la un nivel sofisticat. Aceste modele pot îndeplini o gamă largă de sarcini lingvistice, de la răspunsuri la întrebări și traduceri, până la scrierea de povești și cod.


In [13]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [14]:
PROMPT_RO = """
Analizează, în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani dintr-o perspectivă critică anti-sistem.
Evidențiază tensiunile dintre cetățeni, partidele tradiționale, instituții și elitele politice.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără atacuri personale și fără limbaj extremist.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii cinci ani, politica românească a fost marcată de o luptă continuă între cetățenii care cer transparență și responsabilitate și partidele tradiționale, adesea percepute ca fiind corupte și deconectate de realitățile sociale. Această tensiune a fost amplificată de implicarea instituțiilor și a elitelor politice, care au fost acuzate de menținerea unor interese proprii în detrimentul binelui public.

--- Gemini 2.5 Flash ---
Ultimii cinci ani au marcat o consolidare a puterii partidelor tradiționale, adesea prin alianțe pragmatice, perpetuând un sistem perceput ca rezistent la reforme autentice. Această dinamică a adâncit tensiunile dintre cetățeni, care resimt o lipsă de reprezentare și o perpetuare a clientelismului, și elitele politice, partidele tradiționale și instituțiile percepute ca fiind subordonate intereselor de grup.

--- OpenRouter Free ---
În ultimii 5 ani, politica românească a fost marcată de creșterea tensiunii dintre cetățeni și 

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [15]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice dintr-o perspectivă critică anti-sistem moderată.
Răspunzi scurt, clar și nu inventezi informații.
Identifici neîncrederea față de partide, instituții și elite politice, fără atacuri personale.
"""

PROMPT = """
Analizează următorul comentariu politic:
„Partidele tradiționale își protejează privilegiile, iar cetățenii obișnuiți suportă costurile. Instituțiile par mai interesate de stabilitatea sistemului decât de nevoile oamenilor.”

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Critic, acuzator.
Emoție dominantă: Frustrare, dezamăgire.
Țintă principală: Partidele politice tradiționale și instituțiile statului.
Populism: Da.

--- Gemini 2.5 Flash ---
Ton: Critic, acuzator, deziluzionat.
Emoție dominantă: Frustrare, neîncredere.
Țintă principală: Partidele tradiționale și instituțiile.
Populism: da

--- OpenRouter Free ---
Ton: critic și sceptic  
Emoție dominantă: frustrare  
Țintă principală: partidele tradiționale și instituțiile statului  
Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [16]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic_antisistem",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["critic", "acuzator", "neutru", "ironic", "alarmist"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "neincredere", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string",
                "enum": [
                    "partide_traditionale",
                    "elite_politice",
                    "institutii_stat",
                    "guvern",
                    "parlament",
                    "clasa_politica",
                    "sistem_politic",
                    "alta"
                ]
            },
            "orientare_antisistem": {
                "type": "string",
                "enum": ["absenta", "slaba", "moderata", "puternica"]
            },
            "populism": {
                "type": "boolean"
            },
            "tip_critica": {
                "type": "string",
                "enum": [
                    "coruptie",
                    "privilegii_elite",
                    "lipsa_reprezentare",
                    "ineficienta_institutii",
                    "clientelism",
                    "neclar"
                ]
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "orientare_antisistem",
            "populism",
            "tip_critica",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [17]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'acuzator', 'emotie_dominanta': 'furie', 'tinta_principala': 'clasa_politica', 'orientare_antisistem': 'puternica', 'populism': True, 'tip_critica': 'coruptie', 'explicatie_scurta': 'Comentariul acuză clasa politică de corupție și de ignorare a voinței poporului, sugerând o orientare puternic anti-sistem și populism.'}

--- Gemini 2.5 Flash ---
{'ton': 'acuzator', 'emotie_dominanta': 'furie', 'tinta_principala': 'clasa_politica', 'orientare_antisistem': 'puternica', 'populism': True, 'tip_critica': 'coruptie', 'explicatie_scurta': "Comentariul acuză în mod generalizat toți politicienii de corupție și de ignorarea poporului, folosind un limbaj populist care creează o dihotomie între 'oamenii simpli' și 'politicieni'."}

--- OpenRouter Free ---
{'ton': 'neutru', 'emotie_dominanta': 'neincredere', 'tinta_principala': 'clasa_politica', 'orientare_antisistem': 'absenta', 'populism': True, 'tip_critica': 'coruptie', 'explicatie_scurta': "Comentariu expr

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [18]:
PROMPT_STAB = """
Partidele tradiționale și instituțiile au pierdut încrederea unei părți a cetățenilor.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde dintr-o perspectivă critică anti-sistem moderată, fără atacuri personale și fără limbaj extremist.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Pierderea încrederii în partidele tradiționale și instituții poate semnala o dorință crescândă a cetățenilor pentru o reprezentare politică mai autentică și mai responsabilă, care să răspundă nevoilor reale ale societății, nu doar intereselor de grup. Această situație poate duce la o fragmentare politică sporită și la apariția unor noi forme de mobilizare civică, care contestă status quo-ul și cer o reformă profundă a modului în care este guvernată țara.

temperature=0.7:
Pierderea încrederii în partidele tradiționale și instituții poate semnala o dorință crescută a cetățenilor pentru o reprezentare politică mai autentică și pentru o transparență sporită în procesul decizional. Această situație ar putea conduce la apariția unor noi forțe politice sau la presiuni pentru reformarea profundă a celor existente, cu scopul de a restabili legătura dintre clasa politică și nevoile reale 

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da | da | da | nu | |
| Gemini 2.5 Flash | da  | da | da | da | |
| OpenRouter Free | da | parțial | parțial | - | |
### Decizie
**Model principal ales:**  Gemini 2.5 Flash Lite

**Model de rezervă:**  OpenRouter Free

**Temperature recomandată:**  0.1

**De ce am ales acest model?**  Echipa a ales Gemini 2.5 Flash Lite deoarece a oferit răspunsuri mai clare și mai ușor de înțeles în comparație cu celelalte modele testate. La temperature = 0.1, modelul a fost mai stabil și a produs răspunsuri consecvente, potrivite pentru adnotarea comentariilor politice. OpenRouter Free poate fi folosit ca model de rezervă, în cazul în care modelul principal nu este disponibil.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [17]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales